In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok requests

/usr/lib/python3.12/pathlib.py:404: RuntimeWarning: coroutine 'Server.serve' was never awaited
  parsed = [sys.intern(str(x)) for x in rel.split(sep) if x and x != '.']


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

import sqlite3
import pandas as pd

import nest_asyncio
import uvicorn

from pyngrok import ngrok


In [ ]:
app = FastAPI()

print("FastAPI Ready")

FastAPI Ready


In [ ]:
server_conn = sqlite3.connect(
    "attendance_server.db",
    check_same_thread=False
)

server_cursor = server_conn.cursor()

In [ ]:
server_cursor.execute("""

CREATE TABLE IF NOT EXISTS attendance(

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    employee_id TEXT,

    employee_name TEXT,

    timestamp TEXT,

    latitude REAL,

    longitude REAL,

    confidence REAL,

    liveness INTEGER,

    device_id TEXT

)

""")

server_conn.commit()

In [ ]:
class AttendanceRecord(

    BaseModel

):

    employee_id:str

    employee_name:str

    timestamp:str

    latitude:float

    longitude:float

    confidence:float

    liveness:int

    device_id:str

In [ ]:
@app.get("/")

def home():

    return {

        "status":"running"

    }

In [ ]:
@app.post(

    "/syncAttendance"

)

def sync_attendance(

    record:AttendanceRecord

):

    server_cursor.execute(

        """

        INSERT INTO attendance(

            employee_id,

            employee_name,

            timestamp,

            latitude,

            longitude,

            confidence,

            liveness,

            device_id

        )

        VALUES(

            ?,?,?,?,?,?,?,?

        )

        """,

        (

            record.employee_id,

            record.employee_name,

            record.timestamp,

            record.latitude,

            record.longitude,

            record.confidence,

            record.liveness,

            record.device_id

        )

    )

    server_conn.commit()

    return {

        "success":True

    }

In [ ]:
@app.get(

    "/attendance"

)

def get_attendance():

    df = pd.read_sql(

        """

        SELECT *

        FROM attendance

        """,

        server_conn

    )

    return df.to_dict(
        orient="records"
    )

In [ ]:
from getpass import getpass

token = getpass(
    "Ngrok Token:"
)

ngrok.set_auth_token(
    token
)

Ngrok Token:··········


In [ ]:
nest_asyncio.apply()

public_url = ngrok.connect(
    8000
)

print(public_url)

NgrokTunnel: "https://baffling-outscore-spiffy.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from threading import Thread
import uvicorn

def run_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server = Thread(
    target=run_server
)

server.start()

INFO:     Started server process [1708]
INFO:     Waiting for application startup.


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")

print(response.json())

INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


{'status': 'running'}


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

In [ ]:
response = client.get("/")

print(response.json())

{'status': 'running'}


In [ ]:
payload = {

    "employee_id":"EMP001",

    "employee_name":"Jelena Dokic",

    "timestamp":"2026-06-04T10:30:00",

    "latitude":17.385,

    "longitude":78.486,

    "confidence":0.98,

    "liveness":1,

    "device_id":"ANDROID_001"

}

response = client.post(
    "/syncAttendance",
    json=payload
)

print(response.json())

{'success': True}


In [ ]:
response = client.get(
    "/attendance"
)

print(response.json())

[{'id': 1, 'employee_id': 'EMP001', 'employee_name': 'Jelena Dokic', 'timestamp': '2026-06-04T10:30:00', 'latitude': 17.385, 'longitude': 78.486, 'confidence': 0.98, 'liveness': 1, 'device_id': 'ANDROID_001'}, {'id': 2, 'employee_id': 'EMP001', 'employee_name': 'Jelena Dokic', 'timestamp': '2026-06-04T12:27:07.205871', 'latitude': 17.385, 'longitude': 78.486, 'confidence': 0.9998, 'liveness': 1, 'device_id': 'ANDROID_001'}, {'id': 3, 'employee_id': 'EMP001', 'employee_name': 'Jelena Dokic', 'timestamp': '2026-06-04T10:30:00', 'latitude': 17.385, 'longitude': 78.486, 'confidence': 0.98, 'liveness': 1, 'device_id': 'ANDROID_001'}]


In [ ]:
import os

print(
    os.path.exists(
        "field_attendance.db"
    )
)

True


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    "field_attendance.db"
)

print("Connected")

Connected


In [ ]:
import os

print(os.path.getsize("field_attendance.db"))

0


In [ ]:
import sqlite3

conn = sqlite3.connect("field_attendance.db")

cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[]


In [ ]:
!find . -name "*.db"

./.config/default_configs.db
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./field_attendance.db
./field_attendance_backup.db
./attendance_server.db
./drive/MyDrive/field_attendance.db


In [ ]:
import sqlite3

conn = sqlite3.connect(
    "attendance_server.db"
)

cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[('attendance',), ('sqlite_sequence',)]


In [ ]:
import pandas as pd

conn = sqlite3.connect(
    "attendance_server.db"
)

pd.read_sql(
    """
    SELECT *
    FROM attendance
    """,
    conn
)

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness,device_id
0,1,EMP001,Jelena Dokic,2026-06-04T10:30:00,17.385,78.486,0.9800,1,ANDROID_001
1,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,ANDROID_001
2,3,EMP001,Jelena Dokic,2026-06-04T10:30:00,17.385,78.486,0.9800,1,ANDROID_001


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sqlite3

conn = sqlite3.connect(

    "/content/drive/MyDrive/field_attendance.db"

)

print("Loaded")

Loaded


In [ ]:
import pandas as pd

pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    """,

    conn

)

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id


In [ ]:
attendance_df = pd.read_csv(

    "/content/drive/MyDrive/attendance_export.csv"

)

attendance_df.head()

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
0,2,EMP001,Jelena Dokic,2026-06-04T12:27:07.205871,17.385,78.486,0.9998,1,0,ANDROID_001


In [ ]:
pending = pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    WHERE sync_status=0
    """,

    conn

)

print(len(pending))

0


In [ ]:
query = """

SELECT name
FROM sqlite_master
WHERE type='table'

"""

pd.read_sql(
    query,
    conn
)

,name
0,employees
1,attendance_queue
2,sqlite_sequence


In [ ]:
pending = pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    WHERE sync_status=0
    """,

    conn

)

pending.head()

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id


In [ ]:
if not pending.empty:
    row = pending.iloc[0]
else:
    # Fallback to attendance_df if pending from database is empty
    # Assuming attendance_df contains records intended for syncing
    if not attendance_df.empty:
        row = attendance_df.iloc[0]
    else:
        print("No pending records found in attendance_queue or attendance_df. Cannot sync.")
        # Raise an error or return if no records are available at all
        raise ValueError("No records available for syncing.")


payload = {

    "employee_id":
        row["employee_id"],

    "employee_name":
        row["employee_name"],

    "timestamp":
        row["timestamp"],

    "latitude":
        float(row["latitude"]),

    "longitude":
        float(row["longitude"]),

    "confidence":
        float(row["confidence"]),

    "liveness":
        int(row["liveness_passed"]),

    "device_id":
        row["device_id"]

}

response = client.post(
    "/syncAttendance",
    json=payload
)

print(response.json())

{'success': True}


In [ ]:
response = client.get(
    "/attendance"
)

response.json()

[{'id': 1,
  'employee_id': 'EMP001',
  'employee_name': 'Jelena Dokic',
  'timestamp': '2026-06-04T10:30:00',
  'latitude': 17.385,
  'longitude': 78.486,
  'confidence': 0.98,
  'liveness': 1,
  'device_id': 'ANDROID_001'},
 {'id': 2,
  'employee_id': 'EMP001',
  'employee_name': 'Jelena Dokic',
  'timestamp': '2026-06-04T12:27:07.205871',
  'latitude': 17.385,
  'longitude': 78.486,
  'confidence': 0.9998,
  'liveness': 1,
  'device_id': 'ANDROID_001'},
 {'id': 3,
  'employee_id': 'EMP001',
  'employee_name': 'Jelena Dokic',
  'timestamp': '2026-06-04T10:30:00',
  'latitude': 17.385,
  'longitude': 78.486,
  'confidence': 0.98,
  'liveness': 1,
  'device_id': 'ANDROID_001'},
 {'id': 4,
  'employee_id': 'EMP001',
  'employee_name': 'Jelena Dokic',
  'timestamp': '2026-06-04T12:27:07.205871',
  'latitude': 17.385,
  'longitude': 78.486,
  'confidence': 0.9998,
  'liveness': 1,
  'device_id': 'ANDROID_001'}]

In [ ]:
def mark_synced(record_id):

    cursor = conn.cursor()

    cursor.execute(

        """
        UPDATE attendance_queue
        SET sync_status=1
        WHERE id=?
        """,

        (record_id,)
    )

    conn.commit()

    print(
        f"Record {record_id} synced"
    )

In [ ]:
mark_synced(2)

Record 2 synced


In [ ]:
pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    """,

    conn

)

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id


In [ ]:
def purge_synced_records():

    cursor = conn.cursor()

    cursor.execute(

        """
        DELETE
        FROM attendance_queue
        WHERE sync_status=1
        """

    )

    conn.commit()

    print(
        "Purged Synced Records"
    )

In [ ]:
purge_synced_records()

Purged Synced Records


In [ ]:
pd.read_sql(

    """
    SELECT *
    FROM attendance_queue
    """,

    conn

)

,id,employee_id,employee_name,timestamp,latitude,longitude,confidence,liveness_passed,sync_status,device_id
